In [3]:
import os
import pathlib
from typing import List
from datasets import load_dataset
import evaluate
import torch
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from hf_wrapper import GPTForSequenceClassification
from model import GPT, GPTConfig
from tokenizer import load_tokenizer, eod_token
from sklearn.metrics import precision_score, recall_score, f1_score

# === Config ===
BASE_DIR = pathlib.Path("/fs/scratch/PAS2836/ipa_gpt")
TOKENIZER_DIR = pathlib.Path("/fs/ess/PAS2836/ipa_gpt/tokenizers")

CHECKPOINTS = {
    "ipa": BASE_DIR / "checkpoints/russian_polish_ipa_12_5_50k/ckpt.pt",
    "normal": BASE_DIR / "checkpoints/russian_polish_normal_12_5_50k/ckpt.pt",
}

TOKENIZERS = {
    "ipa": (
        TOKENIZER_DIR / "bpe-rus-pol-ipa-number-preservation-vocab.json",
        TOKENIZER_DIR / "bpe-rus-pol-ipa-number-preservation-merges.txt",
    ),
    "normal": (
        TOKENIZER_DIR / "bpe-rus-pol-normal-number-preservation-vocab.json",
        TOKENIZER_DIR / "bpe-rus-pol-normal-number-preservation-merges.txt",
    ),
}

args = {
    'epochs': 3,
    'context_size': 512,
    'learning_rate': 2e-5,
    'batch_size': 16,
    'hf_cache_dir': pathlib.Path('cache'),
    'device': 'cuda',
}

def load_pretrained_model(path: pathlib.Path, device: str = 'cuda') -> GPT:
    checkpoint = torch.load(path, map_location=device)
    gptconf = GPTConfig(**checkpoint['model_args'])
    model = GPT(gptconf)
    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'
    for k in list(state_dict.keys()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    filtered = {k: v for k, v in state_dict.items()
                if k in model.state_dict() and v.shape == model.state_dict()[k].shape}
    model.load_state_dict({**model.state_dict(), **filtered})
    return model.to(device)

def flatten_single_field(examples, field: str) -> List[str]:
    return examples[field]

def preprocess_function(examples, model_type, field_name, tokenizer):
    inputs = flatten_single_field(examples, field_name)
    return tokenizer(inputs, truncation=True, max_length=args['context_size'])

# === Train on Russian (SentRuEval), Evaluate on Polish (Allegro) ===
for model_type in ['ipa', 'normal']:
    print(f"\n🔤 Loading {model_type.upper()} model and tokenizer...")
    vocab_path, merges_path = TOKENIZERS[model_type]
    tokenizer = load_tokenizer(vocab_path, merges_path)

    base_model = load_pretrained_model(CHECKPOINTS[model_type], args['device'])
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.padding_side = tokenizer.padding_side
    model = GPTForSequenceClassification(base_model, num_classes=2).to(args['device'])

    # === Load datasets ===
    ru_ds = load_dataset("iggy12345/sentirueval2016-ipa", split="train", cache_dir=str(args['hf_cache_dir']))
    pl_ds = load_dataset("iggy12345/allegro-reviews-ipa", split="train", cache_dir=str(args['hf_cache_dir']))
    pl_ds = pl_ds.filter(lambda x: x["binary_label"] != 0)  # remove neutral
    pl_ds = pl_ds.rename_column("binary_label", "label")

    field = "text-phoneme" if model_type == "ipa" else "text"

    train_encoded = ru_ds.map(
    lambda x: preprocess_function(x, model_type, field, tokenizer), batched=True
  )
    eval_encoded = pl_ds.map(
        lambda x: preprocess_function(x, model_type, field, tokenizer), batched=True
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = torch.from_numpy(logits).argmax(dim=-1)
        labels = torch.tensor(labels)
        return {
            "accuracy": (preds == labels).sum().item() / len(labels),
            "precision": precision_score(labels, preds, average="binary", zero_division=0),
            "recall": recall_score(labels, preds, average="binary", zero_division=0),
            "f1": f1_score(labels, preds, average="binary", zero_division=0),
        }

    output_dir = pathlib.Path(f"./outputs_sentiment/ru2pl_{model_type}")
    output_dir.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=1,
        metric_for_best_model="f1",
        load_best_model_at_end=True,
        learning_rate=args['learning_rate'],
        per_device_train_batch_size=args['batch_size'],
        per_device_eval_batch_size=args['batch_size'],
        num_train_epochs=args['epochs'],
        weight_decay=0.01,
        logging_steps=500,
        logging_dir='./logs',
        fp16=True,
        disable_tqdm=False,
        warmup_ratio=0.3,
        save_safetensors=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_encoded,
        eval_dataset=eval_encoded,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print(f"\n🚀 Training {model_type.upper()} model on RU → Evaluating on PL")
    trainer.train()

    results = trainer.evaluate()
    print(f"\n✅ Evaluation results (RU→PL, {model_type.upper()}):\n{results}")



🔤 Loading IPA model and tokenizer...
number of parameters: 123.35M


Map: 100%|██████████| 8369/8369 [00:02<00:00, 3827.20 examples/s]
/tmp/slurmtmp.1600833/ipykernel_3321358/3292186701.py:127: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.



🚀 Training IPA model on RU → Evaluating on PL


wandb: Currently logged in as: orugantikoundinya7 (orugantikoundinya7-ohio-state-buckeyes) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [8,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [14,0,0] Assertion `t >= 0 && t < n_classes` failed.


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import os
import pathlib
from typing import List
from datasets import load_dataset, concatenate_datasets, ClassLabel
import evaluate
import torch
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from hf_wrapper import GPTForSequenceClassification
from model import GPT, GPTConfig
from tokenizer import load_tokenizer, eod_token
from sklearn.metrics import precision_score, recall_score, f1_score

# --- Config ---
BASE_DIR = pathlib.Path("/fs/scratch/PAS2836/ipa_gpt")
TOKENIZER_DIR = pathlib.Path("/fs/ess/PAS2836/ipa_gpt/tokenizers")

CHECKPOINTS = {
    "ipa": BASE_DIR / "checkpoints/russian_polish_ipa_12_5_50k/ckpt.pt",
    "normal": BASE_DIR / "checkpoints/russian_polish_normal_12_5_50k/ckpt.pt",
}

TOKENIZERS = {
    "ipa": (
        TOKENIZER_DIR / "bpe-rus-pol-ipa-number-preservation-vocab.json",
        TOKENIZER_DIR / "bpe-rus-pol-ipa-number-preservation-merges.txt",
    ),
    "normal": (
        TOKENIZER_DIR / "bpe-rus-pol-normal-number-preservation-vocab.json",
        TOKENIZER_DIR / "bpe-rus-pol-normal-number-preservation-merges.txt",
    ),
}

LANG_TO_DATASET = {
    "ru": "iggy12345/xnli-ru-ipa",
    "pl": "iggy12345/cdsc-e-ipa"
}

args = {
    'epochs': 3,
    'context_size': 1024,
    'learning_rate': 2e-5,
    'batch_size': 16,
    'hf_cache_dir': pathlib.Path('cache'),
    'device': 'cuda',
}

def load_pretrained_model(path: pathlib.Path, device: str = 'cuda') -> GPT:
    checkpoint = torch.load(path, map_location=device)
    gptconf = GPTConfig(**checkpoint['model_args'])
    model = GPT(gptconf)
    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'
    for k in list(state_dict.keys()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    filtered = {k: v for k, v in state_dict.items()
                if k in model.state_dict() and v.shape == model.state_dict()[k].shape}
    model.load_state_dict({**model.state_dict(), **filtered})
    return model.to(device)

def flatten_multi_features(examples, features: List[str]) -> List[str]:
    sep = f'\n\n{eod_token}\n\n'
    return [sep.join([x or '' for x in items]) for items in zip(*[examples[f] for f in features])]

def get_fields(model_type, dataset_name):
    if model_type == "ipa":
        if "cdsc" in dataset_name:
            return ["sentence_A-phoneme", "sentence_B-phoneme"]
        else:
            return ["premise-phoneme", "hypothesis-phoneme"]
    else:
        if "cdsc" in dataset_name:
            return ["sentence_A", "sentence_B"]
        else:
            return ["premise", "hypothesis"]

def load_and_preprocess(dataset_name, split, tokenizer, model_type):
    ds = load_dataset(dataset_name, split=split, cache_dir=str(args['hf_cache_dir']))
    if 'label' in ds.features and not isinstance(ds.features['label'], ClassLabel):
        ds = ds.cast_column("label", ClassLabel(names=['entailment', 'neutral', 'contradiction']))
    fields = get_fields(model_type, dataset_name)
    def preprocess(examples):
        features = flatten_multi_features(examples, fields)
        return tokenizer(features, truncation=True, max_length=args['context_size'])
    return ds.map(preprocess, batched=True)

metric = evaluate.load("xnli", "en")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.from_numpy(logits).argmax(dim=-1)
    labels = torch.tensor(labels)
    return {
        "accuracy": (preds == labels).sum().item() / len(labels),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
        "f1": f1_score(labels, preds, average="weighted", zero_division=0),
    }

# === Train on RU+PL, Evaluate on RU+PL ===
for model_type in ['ipa', 'normal']:
    print(f"Starting run for {model_type.upper()} on BOTH (RU+PL)")

    # Load tokenizer and model
    vocab_path, merges_path = TOKENIZERS[model_type]
    tokenizer = load_tokenizer(vocab_path, merges_path)
    base_model = load_pretrained_model(CHECKPOINTS[model_type], args['device'])
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.padding_side = tokenizer.padding_side
    model = GPTForSequenceClassification(base_model, num_classes=3).to(args['device'])

    # Load + preprocess datasets
    train_ru = load_and_preprocess(LANG_TO_DATASET['ru'], 'train', tokenizer, model_type)
    train_pl = load_and_preprocess(LANG_TO_DATASET['pl'], 'train', tokenizer, model_type)
    train_dataset = concatenate_datasets([train_ru, train_pl])

    eval_ru = load_and_preprocess(LANG_TO_DATASET['ru'], 'validation', tokenizer, model_type)
    eval_pl = load_and_preprocess(LANG_TO_DATASET['pl'], 'train[:20%]', tokenizer, model_type)
    eval_dataset = concatenate_datasets([eval_ru, eval_pl])

    # Setup training args
    output_dir = pathlib.Path(f"./training_outputs_rupl/both2both_{model_type}")
    output_dir.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="steps",
        eval_steps=1000,
        save_strategy="steps",
        save_steps=1000,
        save_total_limit=1,
        metric_for_best_model="f1",
        load_best_model_at_end=True,
        learning_rate=args['learning_rate'],
        per_device_train_batch_size=args['batch_size'],
        per_device_eval_batch_size=args['batch_size'],
        num_train_epochs=args['epochs'],
        weight_decay=0.01,
        logging_steps=1000,
        logging_dir='./logs',
        fp16=True,
        disable_tqdm=False,
        warmup_ratio=0.3,
        save_safetensors=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )

    print(f"Training {model_type.upper()} model on RU+PL → Evaluating on RU+PL")
    trainer.train()

    print(f"Final evaluation on RU+PL for {model_type.upper()}")
    results = trainer.evaluate()
    print(results)

In [5]:
from datasets import load_dataset

# Load the Allegro reviews dataset
dataset = load_dataset("iggy12345/sentirueval2016-ipa")

# Function to print examples for a given label
def show_examples(dataset, label_value, num=3):
    print(f"\n🔹 Examples with label {label_value}:\n" + "-"*60)
    filtered = dataset["train"].filter(lambda x: x["label"] == label_value)
    for ex in filtered.select(range(min(num, len(filtered)))):
        print(f"Text: {ex['text']}")
        print("-" * 60)

# Show examples for -1 (negative), 0 (neutral), 1 (positive)
for label in [-1, 0, 1]:
    show_examples(dataset, label)



🔹 Examples with label -1:
------------------------------------------------------------
Text: Блядский пидорский билайн. извините но нет сил
------------------------------------------------------------
Text: Почему МТС в Дагестане не работает? Кто знает?
------------------------------------------------------------
Text: у меня теле2 уже больше 10 лет, так что можешь мне не рассказывать:) в школьном гардеробе он не принимает совсем, а в бассейне плохо.
------------------------------------------------------------

🔹 Examples with label 0:
------------------------------------------------------------


Filter: 100%|██████████| 3000/3000 [00:00<00:00, 158953.42 examples/s]


Text: Райффайзен банк ростов http://t.co/bxtAzFUULF
------------------------------------------------------------
Text: RT @xuxuvoleluly: «Ростелеком» подсчитал ущерб от наводнения на Алтае
------------------------------------------------------------
Text: банки тюмени сбербанк http://t.co/t5OXtLb969
------------------------------------------------------------

🔹 Examples with label 1:
------------------------------------------------------------


Filter: 100%|██████████| 3000/3000 [00:00<00:00, 165032.62 examples/s]

Text: Названы популярные неотраслевые ПИФы России: Универ, Открытие и Газпромбанк
------------------------------------------------------------
Text: RT @Dimka_ji_est: Реклама Билайна о iPhone 5s такая классная!
------------------------------------------------------------
Text: @Damirrrrka @886dmitriev  ДД! Скорее всего LTE у абонентов совместного предприятия Ростелеком и Теле2 появится в 2015 году
------------------------------------------------------------
